# Stage-gate playground — tilearray 0.3

Drive the **five-minute OSM path** before cutting **0.3.0**: edit knobs, `plan_array`, fetch, plot, check EPSG:3857 georef.

```bash
uv sync --dev
uv run jupyter lab   # open this notebook
```

Habit: `config → plan_array (optional) → create_array → compute_mosaic → plot`.


## Setup


In [ ]:
from __future__ import annotations

import time

import matplotlib.pyplot as plt

import tilearray
from tilearray import (
    XYZConfig,
    compute_mosaic,
    create_array,
    fetch_summary,
    inspect_endpoint,
    plan_array,
)
from tilearray.fetch_presets import OSM_TILE_TEMPLATE, osm_fetch_defaults
from tilearray.types import CRS

print("tilearray", tilearray.__version__)


def box_around(x: float, y: float, width: float) -> tuple[float, float, float, float]:
    half = width / 2.0
    return (x - half, y - half, x + half, y + half)


def summarize(arr, *, wall_s: float | None = None) -> None:
    if wall_s is not None:
        print(f"wall_s={wall_s:.2f}")
    print(f"shape={arr.shape}  dtype={arr.dtype}  dims={arr.dims}")
    for key in ("crs", "transform", "bbox_requested", "service_type"):
        if key in arr.attrs:
            print(f"  {key}={arr.attrs[key]!r}")
    print(
        "  extent_m:",
        float(arr.x.min()),
        float(arr.y.min()),
        float(arr.x.max()),
        float(arr.y.max()),
    )


def show(arr, *, title: str = "") -> None:
    # xarray imshow uses mosaic x/y (EPSG:3857 metres for XYZ)
    plotter = arr.squeeze().plot.imshow(cmap="gray", add_colorbar=False)
    if title:
        plotter.axes.set_title(title)
    plt.tight_layout()
    plt.show()


## Knobs — edit these


In [ ]:
# --- OSM five-minute path (edit me) --------------------------------------
LON, LAT = -0.005, 51.500  # central London (quickstart window centre-ish)
# LON, LAT = -0.1276, 51.5072  # Trafalgar-ish
WIDTH_DEG = 0.01  # ~1 km; keep small
ZOOM = 16

MAX_CONCURRENT = None  # e.g. 1; None = osm_fetch_defaults()
SHOW_PLAN = True
RUN_INSPECT = False  # optional: inspect XYZ template limits


## Optional — inspect template


In [ ]:
if RUN_INSPECT:
    inspection = inspect_endpoint(OSM_TILE_TEMPLATE)
    print(inspection)
    print(inspection.build_config(zoom=ZOOM))


## Run — plan, fetch, plot


In [ ]:
bbox = box_around(LON, LAT, WIDTH_DEG)
crs = CRS.EPSG_4326  # query CRS (lon/lat)

extra = {}
if MAX_CONCURRENT is not None:
    extra["max_concurrent_requests"] = MAX_CONCURRENT

config = XYZConfig.from_url(
    OSM_TILE_TEMPLATE,
    zoom=ZOOM,
    **osm_fetch_defaults(),
    **extra,
)

print(f"query bbox lon/lat={bbox}  zoom={ZOOM}")

if SHOW_PLAN:
    plan = plan_array(config, bbox, crs)
    print(plan)

da = create_array(config, bbox, crs)
t0 = time.perf_counter()
arr = compute_mosaic(da)
summarize(arr, wall_s=time.perf_counter() - t0)
print(fetch_summary(da))

crs_out = str(arr.attrs.get("crs", "")).upper()
assert crs_out in {"EPSG:3857", "3857"}, (
    f"expected mosaic CRS EPSG:3857, got {arr.attrs.get('crs')!r}"
)
print("georef ok: mosaic CRS is EPSG:3857 (query was lon/lat)")

show(arr, title="OSM XYZ  © OpenStreetMap contributors")


## Notes

- `crs=` is **query** only for XYZ; mosaic `attrs["crs"]` is **EPSG:3857** over the tile union (metres).
- OSM preset ceiling is 2 — do not raise past it for stage-gate.
- Prefer lazy + `compute_mosaic`; `load_array` is eager sugar.
- Broader multi-layer pad: `stage_gate_0_2.ipynb`.

### What felt wrong?

- [ ] Five-minute path clear?
- [ ] Plan / fetch_summary useful?
- [ ] 3857 georef readable on plot / attrs?
- [ ] Anything blocking a **0.3.0** cut?

```
(your notes)
```
